In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
DATA_DIR = Path("../Data/raw")

users = pd.read_csv(
    DATA_DIR / "users.csv",
    parse_dates=["signup_date"]
)

products = pd.read_csv(
    DATA_DIR / "products.csv",
    parse_dates=["launch_date"]
)

assignments = pd.read_csv(
    DATA_DIR / "experiment_assignment.csv"
)

sessions = pd.read_csv(
    DATA_DIR / "sessions.csv",
    parse_dates=["session_start", "session_end"]
)

events = pd.read_csv(
    DATA_DIR / "events.csv",
    parse_dates=["event_time"]
)

orders = pd.read_csv(
    DATA_DIR / "orders.csv",
    parse_dates=["purchase_time"]
)

In [4]:
for df in [
    users,
    products,
    assignments,
    sessions,
    events,
    orders
]:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
    )

In [5]:
for df in [users, sessions, orders]:

    for col in df.columns:

        if df[col].dtype == "object":

            values = set(
                df[col]
                .dropna()
                .astype(str)
                .str.lower()
                .unique()
            )

            if values.issubset(
                {"true", "false"}
            ):
                df[col] = (
                    df[col]
                    .astype(str)
                    .str.lower()
                    .map({
                        "true": True,
                        "false": False
                    })
                )

In [6]:
events["is_homepage"] = (
    events["event_name"]
    .eq("Homepage")
)

events["is_search"] = (
    events["event_name"]
    .eq("Search")
)

events["is_product_view"] = (
    events["event_name"]
    .eq("Product_View")
)

events["is_add_cart"] = (
    events["event_name"]
    .eq("Add_Cart")
)

events["is_checkout"] = (
    events["event_name"]
    .eq("Checkout")
)

events["is_shipping"] = (
    events["event_name"]
    .eq("Shipping")
)

events["is_payment"] = (
    events["event_name"]
    .isin([
        "Payment",
        "One_Click_Payment"
    ])
)

events["is_purchase"] = (
    events["event_name"]
    .isin([
        "Purchase",
        "Order_Confirmed"
    ])
)

In [8]:
events = events.merge(
    sessions[["session_id", "user_id"]],
    on="session_id",
    how="left"
)

In [9]:
events["is_homepage"] = (
    events["event_name"] == "Homepage"
).astype(int)

events["is_search"] = (
    events["event_name"] == "Search"
).astype(int)

events["is_product_view"] = (
    events["event_name"] == "Product_View"
).astype(int)

events["is_add_cart"] = (
    events["event_name"] == "Add_to_Cart"
).astype(int)

events["is_checkout"] = (
    events["event_name"] == "Checkout"
).astype(int)

events["is_shipping"] = (
    events["event_name"] == "Shipping"
).astype(int)

events["is_payment"] = (
    events["event_name"].isin(
        ["Payment", "One_Click_Payment"]
    )
).astype(int)

events["is_purchase"] = (
    events["event_name"] == "Purchase"
).astype(int)

In [10]:
user_event_metrics = (
    events
    .groupby("user_id")
    .agg(
        total_events=("event_id", "count"),
        homepage_views=("is_homepage", "sum"),
        searches=("is_search", "sum"),
        product_views=("is_product_view", "sum"),
        add_to_carts=("is_add_cart", "sum"),
        checkout_starts=("is_checkout", "sum"),
        shipping_events=("is_shipping", "sum"),
        payment_events=("is_payment", "sum"),
        purchase_events=("is_purchase", "sum")
    )
    .reset_index()
)

In [11]:
session_metrics = (
    sessions
    .groupby("user_id")
    .agg(
        total_sessions=("session_id", "count"),
        total_session_duration=("session_duration", "sum"),
        avg_session_duration=("session_duration", "mean"),
        max_session_duration=("session_duration", "max"),
        total_pages_viewed=("pages_viewed", "sum"),
        avg_pages_viewed=("pages_viewed", "mean"),
        bounce_sessions=("bounce", "sum")
    )
    .reset_index()
)

In [12]:
session_metrics["bounce_rate"] = (
    session_metrics["bounce_sessions"]
    /
    session_metrics["total_sessions"]
)

In [13]:
orders["is_completed"] = (
    orders["order_status"]
    .eq("Completed")
)

In [14]:
order_metrics = (
    orders
    .groupby("user_id")
    .agg(
        total_orders=("order_id", "count"),
        completed_orders=("is_completed", "sum"),
        total_quantity=("quantity", "sum"),
        total_revenue=("total_amount", "sum"),
        total_profit=("profit", "sum"),
        total_discount=("discount", "sum"),
        avg_order_value=("total_amount", "mean"),
        avg_profit_per_order=("profit", "mean")
    )
    .reset_index()
)

In [15]:
user_metrics = (
    users
    .merge(
        assignments,
        on="user_id",
        how="left"
    )
    .merge(
        user_event_metrics,
        on="user_id",
        how="left"
    )
    .merge(
        session_metrics,
        on="user_id",
        how="left"
    )
    .merge(
        order_metrics,
        on="user_id",
        how="left"
    )
)

In [16]:
metric_columns = [
    "total_events",
    "homepage_views",
    "searches",
    "product_views",
    "add_to_carts",
    "checkout_starts",
    "shipping_events",
    "payment_events",
    "purchase_events",
    "total_sessions",
    "total_session_duration",
    "avg_session_duration",
    "max_session_duration",
    "total_pages_viewed",
    "avg_pages_viewed",
    "bounce_sessions",
    "bounce_rate",
    "total_orders",
    "completed_orders",
    "total_quantity",
    "total_revenue",
    "total_profit",
    "total_discount",
    "avg_order_value",
    "avg_profit_per_order"
]

user_metrics[metric_columns] = (
    user_metrics[metric_columns]
    .fillna(0)
)

In [17]:
user_metrics["converted"] = (
    user_metrics["completed_orders"] > 0
)

In [18]:
user_metrics["carted"] = (
    user_metrics["add_to_carts"] > 0
)

In [19]:
user_metrics["started_checkout"] = (
    user_metrics["checkout_starts"] > 0
)

In [20]:
user_metrics["reached_payment"] = (
    user_metrics["payment_events"] > 0
)

In [21]:
user_metrics["cart_rate"] = np.where(
    user_metrics["product_views"] > 0,
    user_metrics["carted"] /
    user_metrics["product_views"],
    0
)

In [22]:
user_metrics["revenue_per_user"] = (
    user_metrics["total_revenue"]
)

In [23]:
user_metrics["profit_per_user"] = (
    user_metrics["total_profit"]
)

In [24]:
OUTPUT_DIR = Path(
    "../outputs/metrics"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

user_metrics.to_csv(
    OUTPUT_DIR / "user_metrics.csv",
    index=False
)

In [25]:
experiment_metrics = (
    user_metrics
    .groupby("group")
    .agg(
        users=("user_id", "nunique"),
        converted_users=("converted", "sum"),
        sessions=("total_sessions", "sum"),
        product_views=("product_views", "sum"),
        add_to_carts=("add_to_carts", "sum"),
        checkouts=("checkout_starts", "sum"),
        purchases=("completed_orders", "sum"),
        revenue=("total_revenue", "sum"),
        profit=("total_profit", "sum")
    )
    .reset_index()
)

In [26]:
experiment_metrics["conversion_rate"] = (
    experiment_metrics["converted_users"]
    /
    experiment_metrics["users"]
)

In [27]:
experiment_metrics["add_to_cart_rate"] = (
    experiment_metrics["add_to_carts"]
    /
    experiment_metrics["product_views"]
)

In [28]:
experiment_metrics["revenue_per_user"] = (
    experiment_metrics["revenue"]
    /
    experiment_metrics["users"]
)

In [29]:
experiment_metrics["profit_per_user"] = (
    experiment_metrics["profit"]
    /
    experiment_metrics["users"]
)

In [30]:
experiment_metrics["aov"] = np.where(
    experiment_metrics["purchases"] > 0,
    experiment_metrics["revenue"]
    /
    experiment_metrics["purchases"],
    0
)

In [31]:
experiment_metrics.to_csv(
    OUTPUT_DIR / "experiment_metrics.csv",
    index=False
)

experiment_metrics

,group,users,converted_users,sessions,product_views,add_to_carts,checkouts,purchases,revenue,profit,conversion_rate,add_to_cart_rate,revenue_per_user,profit_per_user,aov
0,Control,60000,16039,330509,157490,0,38476,18862.0,12187391.07,3281407.27,0.267317,0.0,203.123185,54.690121,646.134613
1,Treatment,60000,17477,329351,156709,0,38274,21059.0,13604515.99,3646881.08,0.291283,0.0,226.741933,60.781351,646.019089
